In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/mario-trained-model/best_mario_model.pth


# Instalar GYM super mario bros
Se instala la librería gym-super-mario-bros, que permite simular el juego de Super Mario Bros como un entorno compatible con OpenAI Gym

In [2]:
!pip install gym-super-mario-bros

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.1 MB/s eta 0:00:00
  Created wheel for nes-py: filename=nes_py-8.2.1-cp311-cp311-linux_x86_64.whl size=535718 sha256=e42f2093678bf773a8d23b97aa76909501a31282980b0dbf7a97efc75be51fe3
  Stored in directory: /root/.cache/pip/wheels/be/b4/5a/68b9155f1d2380af0e359c71efd4c70518555be4c2f577f1d3
Successfully built nes-py


# Importación de librerias
Este bloque importa todas las librerías necesarias para entrenar un agente de aprendizaje por refuerzo que juega Super Mario Bros

In [3]:
#Importación de librerias
from nes_py.wrappers import JoypadSpace
import gym_super_mario_bros
import gym
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT

# === Deep Learning y redes neuronales ===
import torch                              # Operaciones tensoriales y uso de GPU
from torch import nn                      # Construcción de redes neuronales (nn.Module, etc.)

# === Preprocesamiento de imágenes ===
from torchvision import transforms as T   # Transformaciones: ToTensor, Resize, Normalize, etc.

# === Utilidades del sistema y control de ejecución ===
import os                                 # Operaciones del sistema (crear carpetas, rutas)
import time                               # Medir tiempos, delays
import datetime                           # Manejo de fechas y horas
import copy                               # Copias profundas de objetos (para evitar referencias)
import random                             # Aleatoriedad (acciones, muestreo)

# === Visualización de métricas ===
import matplotlib.pyplot as plt           # Graficar rewards, curvas de pérdida, etc.

# === Procesamiento de imágenes ===
from PIL import Image                     # Manipulación de imágenes (si se requiere)

# === Computación numérica ===
import numpy as np                        # Arreglos, estadísticas, operaciones numéricas

# === Manejo de archivos y buffers ===
from pathlib import Path                  # Manejo moderno y multiplataforma de rutas
from collections import deque             # Buffer de experiencia (Replay Buffer)

# === Entorno Gym (OpenAI) y wrappers ===
import gym                                # Base para entornos de RL
from gym.spaces import Box                # Espacio de observación (usualmente imágenes)
from gym.wrappers import FrameStack       # Apilar múltiples frames (memoria para el agente)

# === Entorno de Mario Bros ===
from nes_py.wrappers import JoypadSpace   # Wrapper que permite configurar controles (combinaciones de botones)
import gym_super_mario_bros               # Carga el entorno SuperMarioBros-v0



# Inicialización del entorno de Super Mario Bros
Se crea el entorno del juego "SuperMarioBros-v0" y se le aplica el wrapper JoypadSpace, que restringe las acciones a un conjunto simple (SIMPLE_MOVEMENT) para facilitar el entrenamiento del agente. Luego, se realiza un paso en el entorno con la acción 0 y se imprimen:
- la forma del estado observado,
- la recompensa recibida,
- el indicador de si el episodio terminó,
- y la información adicional del entorno.

In [4]:
# Inicializar entorno de super mario bros
env = gym_super_mario_bros.make("SuperMarioBros-v0")

env = JoypadSpace(env, SIMPLE_MOVEMENT)

env.reset()
next_state, reward, done, info = env.step(action=0)
print(f"{next_state.shape},\n {reward},\n {done},\n {info}")

/usr/local/lib/python3.11/dist-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment SuperMarioBros-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(


(240, 256, 3),
 0.0,
 False,
 {'coins': 0, 'flag_get': False, 'life': 2, 'score': 0, 'stage': 1, 'status': 'small', 'time': 400, 'world': 1, 'x_pos': 40, 'y_pos': 79}


/usr/local/lib/python3.11/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.11/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.11/dist-packages/gym/utils/passive_env_checker.py:227: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(
/usr/local/lib/python3.11/dist-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated

# Preprocesar Entorno
Se crean Clases y wrappers personalizados para preprocesar el entorno de Super Mario Bros: SkipFrame repite acciones por varios frames acumulando la recompensa; GrayScaleObservation convierte las imágenes a escala de grises normalizadas; ResizeObservation redimensiona las imágenes; finalmente, FrameStack apila múltiples frames para dar contexto temporal al agente

In [5]:
class SkipFrame(gym.Wrapper):
    """
        SkipFrame wrapper: Repite una acción por N frames y acumula las recompensas.
    """
    def __init__(self, env, skip):
        super().__init__(env)
        self._skip = skip
        
    def step(self, action):
        total_reward = 0.0
        done = False
        for i in range(self._skip):
            obs, reward, done, info = self.env.step(action)
            total_reward += reward
            if done:
                break
        return obs, total_reward, done, info

class GrayScaleObservation(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        obs_shape = self.observation_space.shape[:2]
        self.observation_space = Box(low=0, high=1.0, shape=(1,) + obs_shape, dtype=np.float32)
        self.transform = T.Grayscale()
    
    def permute_orientation(self, observation):
        # (H, W, C) → (C, H, W)
        observation = np.transpose(observation, (2, 0, 1))
        return torch.tensor(observation.copy(), dtype=torch.float32) / 255.0

    def observation(self, observation):
        observation = self.permute_orientation(observation)
        return self.transform(observation)


class ResizeObservation(gym.ObservationWrapper):
    def __init__(self, env, shape):
        super().__init__(env)
        if isinstance(shape, int):
            self.shape = (shape, shape)
        else:
            self.shape = tuple(shape)

        c = self.observation_space.shape[0]
        self.observation_space = Box(low=0.0, high=1.0, shape=(c, *self.shape), dtype=np.float32)

        self.transform = T.Resize(self.shape)

    def observation(self, observation):
        return self.transform(observation)


# Preprocesar entorno
env = SkipFrame(env, skip=4)
env = GrayScaleObservation(env)
env = ResizeObservation(env, shape=84)
env = FrameStack(env, num_stack=4)

# Redes Neuronales Profundas con Doble Q-Learning
Se crea una Red neuronal convolucional MarioNet para estimar valores Q: recibe imágenes de tamaño (4, 84, 84), aplica tres capas convolucionales con ReLU, seguido de capas densas y salida final con el valor Q para cada acción. Contiene dos redes (online y target) idénticas; la red target está congelada (no entrenable) para usar en la actualización estable del algoritmo DQN.

In [6]:

class MarioNet(nn.Module):
    """
        input -> (conv2d + relu) x 3 -> flatten -> (dense + relu) x 2 -> output
    """
    def __init__(self, input_dim, output_dim):
        super().__init__()
        c, w, h = input_dim
        
        if h != 84: raise ValueError(f"Expecting input height: 84, got: {h}")
        if w != 84: raise ValueError(f"Expecting input width: 84, got: {w}")
            
        self.online = nn.Sequential(
            nn.Conv2d(in_channels=c, out_channels=32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, output_dim)
        )
        
        self.target = copy.deepcopy(self.online)
        
        # freeze Q-target parameters
        for p in self.target.parameters():
            p.requires_grad = False
            
    def forward(self, input, model):
        if model == "online":
            return self.online(input)
        elif model == "target":
            return self.target(input)

# Agente
Define el agente que controla a Mario usando un enfoque epsilon-greedy: al principio explora aleatoriamente y, con el tiempo, explota lo aprendido usando la red neuronal MarioNet. Implementa:

- act: selección de acción con política epsilon-greedy.

- cache: almacenamiento de experiencias en un buffer de repetición (deque).

- recall: muestreo aleatorio de un minibatch del buffer para entrenamiento.

La clase también maneja el decaimiento de la tasa de exploración y permite ejecución en GPU si está disponible.

In [7]:
class Mario:
    '''
        Mario randomly explores with a chance of self.exploration_rate.
        When he chooses to exploit, he relies on MarioNet to provide the most optimal action.
    '''
    def __init__(self, state_dim, action_dim, use_cuda):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.use_cuda = use_cuda
        self.memory = deque(maxlen=20000)
        self.batch_size = 16
        
        self.net = MarioNet(self.state_dim, self.action_dim).float()
        if self.use_cuda:
            self.net = self.net.to(device="cuda")
        
        self.exploration_rate = 1
        self.exploration_rate_decay = 0.99999975
        self.exploration_rate_min = 0.1
        self.curr_step = 0

        self.save_every = 5e5
    
    
    def act(self, state):
        '''
            Given a state, choose an epsilon-greedy action and update value of step.
        '''
        # Exploration
        if np.random.rand() < self.exploration_rate:
            action_idx = np.random.randint(self.action_dim)
    
        # Exploitation
        else:
            state = torch.tensor(state.__array__(), device=self.device).unsqueeze(0)  # [1, 4, 1, 84, 84]
            if state.ndim == 5 and state.shape[2] == 1:
                state = state.squeeze(2)  # → [1, 4, 84, 84]
            action_values = self.net(state, model="online")
            action_idx = torch.argmax(action_values, axis=1).item()
    
        # Decrease exploration_rate
        self.exploration_rate *= self.exploration_rate_decay
        self.exploration_rate = max(self.exploration_rate_min, self.exploration_rate)
    
        # Increment step
        self.curr_step += 1
        return action_idx

    
    
    def cache(self, state, next_state, action, reward, done):
        """
            Store the experience to self.memory (replay buffer).
        """
        # Convertir a numpy uint8 comprimido
        state = (state.__array__() * 255).astype(np.uint8)
        next_state = (next_state.__array__() * 255).astype(np.uint8)
    
        # Guardar solo números puros
        self.memory.append((state, next_state, int(action), float(reward), bool(done)))

        
    def recall(self):
        batch = random.sample(self.memory, self.batch_size)
        states, next_states, actions, rewards, dones = zip(*batch)
    
        # Reconstruir tensores desde numpy y normalizarlos
        state = torch.tensor(np.array(states) / 255, dtype=torch.float32)
        next_state = torch.tensor(np.array(next_states) / 255, dtype=torch.float32)
        action = torch.tensor(actions)
        reward = torch.tensor(rewards)
        done = torch.tensor(dones)
    
        if self.use_cuda:
            state = state.cuda()
            next_state = next_state.cuda()
            action = action.cuda()
            reward = reward.cuda()
            done = done.cuda()
    
        return state, next_state, action, reward, done

# Estimación TD y Objetivo TD (Diferencia temporal)

Amplía la clase base Mario para incorporar el ciclo de aprendizaje del agente DQN. Incluye:

- Parámetros como gamma (descuento), burnin (inicio de entrenamiento), learn_every (frecuencia de aprendizaje), y sync_every (frecuencia de sincronización de redes).

- td_estimate: estima el valor Q actual desde la red online.

- td_target: calcula el objetivo de aprendizaje usando la red target (Double DQN).

- update_Q_online: actualiza los pesos de la red online usando retropropagación y pérdida Huber (SmoothL1Loss).

- sync_Q_target: sincroniza la red target con la online.

- learn: ejecuta un paso de aprendizaje si se cumplen las condiciones (burn-in y frecuencia), y retorna el valor Q medio estimado y la pérdida.

In [8]:
class Mario(Mario):
    def __init__(self, state_dim, action_dim, use_cuda):
        super().__init__(state_dim, action_dim, use_cuda)
        self.device = torch.device("cuda" if use_cuda else "cpu")
        self.net = self.net.to(self.device)
        self.gamma = 0.9
        self.burnin = 1e4
        self.learn_every = 6
        self.sync_every = 1e4
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=0.00025)
        self.loss_fn = torch.nn.SmoothL1Loss()


        
    def td_estimate(self, state, action):
        current_Q = self.net(state, model="online")[
            np.arange(0, self.batch_size), action
        ]
        return current_Q

    
    @torch.no_grad()
    def td_target(self, reward, next_state, done):
        next_state_Q = self.net(next_state, model="online")
        best_action = torch.argmax(next_state_Q, axis=1)
        next_Q = self.net(next_state, model="target")[
            np.arange(0, self.batch_size), best_action
        ]
        return (reward + (1 - done.float()) * self.gamma * next_Q).float()
    
    
    def update_Q_online(self, td_estimate, td_target):
        loss = self.loss_fn(td_estimate, td_target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    
    def sync_Q_target(self):
        self.net.target.load_state_dict(self.net.online.state_dict())
        
    
    def learn(self):
        if self.curr_step % self.sync_every == 0:
            self.sync_Q_target()
        # if self.curr_step % self.save_every == 0: self.save()
        if self.curr_step < self.burnin:
            return None, None
        if self.curr_step % self.learn_every != 0:
            return None, None
    
        # Sample from memory
        state, next_state, action, reward, done = self.recall()
    
        # Mover a GPU
        state = state.to(self.device)
        next_state = next_state.to(self.device)
        action = action.to(self.device)
        reward = reward.to(self.device)
        done = done.to(self.device)
    
        # 🧼 Eliminar canal adicional [C=1] si existe
        if state.ndim == 5 and state.shape[2] == 1:
            state = state.squeeze(2)  # → [batch, 4, 84, 84]
        if next_state.ndim == 5 and next_state.shape[2] == 1:
            next_state = next_state.squeeze(2)
    
        # Get TD Estimate
        td_est = self.td_estimate(state, action)
    
        # Get TD Target
        td_tgt = self.td_target(reward, next_state, done)
    
        # Backpropagate loss through Q_online
        loss = self.update_Q_online(td_est, td_tgt)
    
        return (td_est.mean().item(), loss)


# Generación de registros

- Registra y reporta métricas clave durante el entrenamiento del agente, incluyendo:

- Recompensa total por episodio (ep_rewards)

- Longitud del episodio (ep_lengths)

- Pérdida promedio y valores Q estimados (ep_avg_losses, ep_avg_qs)

- Promedios móviles de las métricas (últimos 100 episodios)

- Tiempos entre registros para monitorear rendimiento

Implementa:

- log_step: acumula recompensas, pérdidas y valores Q por paso

- log_episode: guarda métricas del episodio completo

- record: imprime un resumen estadístico cada cierto número de episodios

- init_episode: reinicia contadores por episodio

Permite analizar el progreso del aprendizaje del agente de forma continua y estructurada.

In [9]:
class MetricLogger:
    def __init__(self):
        # history metrics
        self.ep_rewards = []
        self.ep_lengths = []
        self.ep_avg_losses = []
        self.ep_avg_qs = []

        # moving averages, added for every call to record()
        self.moving_avg_ep_rewards = []
        self.moving_avg_ep_lengths = []
        self.moving_avg_ep_avg_losses = []
        self.moving_avg_ep_avg_qs = []

        # current episode metric
        self.init_episode()

        # timing
        self.record_time = time.time()

    def log_step(self, reward, loss, q):
        self.curr_ep_reward += reward
        self.curr_ep_length += 1
        if loss:
            self.curr_ep_loss += loss
            self.curr_ep_q += q
            self.curr_ep_loss_length += 1

    def log_episode(self):
        self.ep_rewards.append(self.curr_ep_reward)
        self.ep_lengths.append(self.curr_ep_length)
        if self.curr_ep_loss_length == 0:
            ep_avg_loss = 0
            ep_avg_q = 0
        else:
            ep_avg_loss = np.round(self.curr_ep_loss / self.curr_ep_loss_length, 5)
            ep_avg_q = np.round(self.curr_ep_q / self.curr_ep_loss_length, 5)
        self.ep_avg_losses.append(ep_avg_loss)
        self.ep_avg_qs.append(ep_avg_q)

        self.init_episode()

    def init_episode(self):
        self.curr_ep_reward = 0.0
        self.curr_ep_length = 0
        self.curr_ep_loss = 0.0
        self.curr_ep_q = 0.0
        self.curr_ep_loss_length = 0

    def record(self, episode, epsilon, step):
        mean_ep_reward = np.round(np.mean(self.ep_rewards[-100:]), 3)
        mean_ep_length = np.round(np.mean(self.ep_lengths[-100:]), 3)
        mean_ep_loss = np.round(np.mean(self.ep_avg_losses[-100:]), 3)
        mean_ep_q = np.round(np.mean(self.ep_avg_qs[-100:]), 3)
        self.moving_avg_ep_rewards.append(mean_ep_reward)
        self.moving_avg_ep_lengths.append(mean_ep_length)
        self.moving_avg_ep_avg_losses.append(mean_ep_loss)
        self.moving_avg_ep_avg_qs.append(mean_ep_q)

        last_record_time = self.record_time
        self.record_time = time.time()
        time_since_last_record = np.round(self.record_time - last_record_time, 3)

        print(
            "Episode:{:4d}  :: Step:{:5d}  :: Epsilon:{:8.3f}  :: Mean_Reward:{:8.3f}  :: " \
            "Mean_Length:{:8.3f}  :: Mean_Loss:{:4.3f}  :: Mean_Q_Value:{:8.3f}  :: " \
            "Time_Delta:{:8.3f} "
            .format(episode, step, epsilon, mean_ep_reward, mean_ep_length, 
                    mean_ep_loss, mean_ep_q, time_since_last_record)
        )

# Entrenamiento
Entrena al agente Mario durante 2001 episodios en el entorno Super Mario Bros. En cada episodio:

- Se reinicia el entorno y se interactúa paso a paso con acciones seleccionadas por el agente.

- Las transiciones se almacenan en memoria y se usa aprendizaje off-policy (DQN).

- Se registran métricas de recompensa, pérdida y valores Q.

- Cada 25 episodios:

    - Se imprimen métricas promediadas.

    - Se guarda el modelo si mejora el reward promedio de los últimos 100 episodios.

    - Se limpia la memoria GPU con torch.cuda.empty_cache() y el recolector de basura gc.collect().

Este ciclo permite entrenar, monitorear y guardar automáticamente el mejor modelo alcanzado durante el entrenamiento.

In [10]:
'''
import gc
use_cuda = torch.cuda.is_available()
print(f"Using CUDA: {use_cuda}")
print()

mario = Mario(state_dim=(4, 84, 84), action_dim=env.action_space.n, use_cuda=use_cuda)
logger = MetricLogger()

best_mean_reward = -float('inf')  # Inicializamos el mejor reward con un valor muy bajo

episodes = 2001
for e in range(episodes):

    # Reinicia el entorno
    state = env.reset()
    if isinstance(state, tuple):  # Compatibilidad por si devuelve (obs, info)
        state = state[0]

    while True:
        # Acción del agente
        action = mario.act(state)

        # El entorno responde
        next_state, reward, done, info = env.step(action)
        if isinstance(next_state, tuple):
            next_state = next_state[0]

        # Almacena transición
        mario.cache(state, next_state, action, reward, done)

        # Aprende
        q, loss = mario.learn()

        # Logging
        logger.log_step(reward, loss, q)

        # Actualiza estado
        state = next_state

        # Condición de finalización
        if done or info.get("flag_get", False):
            break

    # Log por episodio
    logger.log_episode()

    if e % 25 == 0:
        logger.record(episode=e, epsilon=mario.exploration_rate, step=mario.curr_step)

        mean_reward = np.mean(logger.ep_rewards[-100:])

        if mean_reward > best_mean_reward:
            best_mean_reward = mean_reward
            torch.save({
                'model_state_dict': mario.net.state_dict(),
                'episode_rewards': logger.ep_rewards
            }, '/kaggle/working/best_mario_model.pth')

            print(f"✅ Modelo guardado en episodio {e} con reward promedio {best_mean_reward:.2f}")

        torch.cuda.empty_cache()
        gc.collect()
        '''

'\nimport gc\nuse_cuda = torch.cuda.is_available()\nprint(f"Using CUDA: {use_cuda}")\nprint()\n\nmario = Mario(state_dim=(4, 84, 84), action_dim=env.action_space.n, use_cuda=use_cuda)\nlogger = MetricLogger()\n\nbest_mean_reward = -float(\'inf\')  # Inicializamos el mejor reward con un valor muy bajo\n\nepisodes = 2001\nfor e in range(episodes):\n\n    # Reinicia el entorno\n    state = env.reset()\n    if isinstance(state, tuple):  # Compatibilidad por si devuelve (obs, info)\n        state = state[0]\n\n    while True:\n        # Acción del agente\n        action = mario.act(state)\n\n        # El entorno responde\n        next_state, reward, done, info = env.step(action)\n        if isinstance(next_state, tuple):\n            next_state = next_state[0]\n\n        # Almacena transición\n        mario.cache(state, next_state, action, reward, done)\n\n        # Aprende\n        q, loss = mario.learn()\n\n        # Logging\n        logger.log_step(reward, loss, q)\n\n        # Actualiza

In [11]:
'''
from IPython.display import FileLink

print("📦 Modelo guardado en /kaggle/working/")
FileLink('/kaggle/working/best_mario_model.pth')
'''

'\nfrom IPython.display import FileLink\n\nprint("📦 Modelo guardado en /kaggle/working/")\nFileLink(\'/kaggle/working/best_mario_model.pth\')\n'

# **Cargar el modelo**
Inicializa el agente Mario, carga directamente los pesos previamente entrenados desde un archivo .pth, los asigna a la red neuronal (load_state_dict) y la pone en modo evaluación (eval()), indicando que el modelo ya está listo para inferencia sin seguir entrenando.

In [12]:
# Inicializar modelo como antes
mario = Mario(state_dim=(4, 84, 84), action_dim=7, use_cuda=torch.cuda.is_available())

# Cargar archivo con pesos directamente
state_dict = torch.load("/kaggle/input/mario-trained-model/best_mario_model.pth", map_location=mario.device)

# Cargar pesos
mario.net.load_state_dict(state_dict)
mario.net.eval()

print("✅ Modelo cargado correctamente.")

✅ Modelo cargado correctamente.


Instala y configura pyvirtualdisplay para simular una pantalla virtual en entornos sin interfaz gráfica (como Kaggle o Colab), permitiendo grabar y visualizar videos de entornos gym. También importa utilidades para mostrar el video (HTML, IPython.display) y para grabarlo (VideoRecorder). La pantalla virtual se inicia con una resolución de 600×300 píxeles.

In [13]:
!pip --disable-pip-version-check install -q pyvirtualdisplay

from pyvirtualdisplay import Display

from IPython import display as ipythondisplay
from IPython.display import HTML

from gym.wrappers.monitoring.video_recorder import VideoRecorder
from glob import glob

import base64
import io

display = Display(visible=0, size=(600, 300))
display.start()

# **Grabar video del agente entrenado**
Graba y muestra un video del agente Mario ejecutando un episodio en el entorno. Utiliza VideoRecorder para capturar cada frame durante el episodio y guarda el video en video_trained.mp4. Luego, convierte el archivo en base64 para incrustarlo como video HTML directamente en el notebook usando IPython.display

In [14]:
import base64, io
from IPython.display import HTML, display
from gym.wrappers.monitoring.video_recorder import VideoRecorder

# Ruta del video
video_path = "/kaggle/working/video_trained.mp4"
recorder = VideoRecorder(env, video_path)

# Ejecutar episodio
state = env.reset()
if isinstance(state, tuple): state = state[0]
recorder.capture_frame()

while True:
    action = mario.act(state)
    next_state, reward, done, info = env.step(action)
    if isinstance(next_state, tuple): next_state = next_state[0]
    recorder.capture_frame()
    state = next_state
    if done or info.get("flag_get", False):
        break

recorder.close()


# Mostrar el video
def show_video_trained(path):
    with io.open(path, 'r+b') as f:
        video = f.read()
    encoded = base64.b64encode(video).decode('ascii')
    return HTML(f"""
        <video width=480 controls autoplay loop>
            <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
        </video>
    """)
    

display(show_video_trained(video_path))


/usr/local/lib/python3.11/dist-packages/gym/wrappers/monitoring/video_recorder.py:67: DeprecationWarning: WARN: `env.metadata["render.modes"] is marked as deprecated and will be replaced with `env.metadata["render_modes"]` see https://github.com/openai/gym/pull/2654 for more details
  logger.deprecation(
/usr/local/lib/python3.11/dist-packages/gym/wrappers/monitoring/video_recorder.py:78: DeprecationWarning: WARN: Recording ability for environment SuperMarioBros-v0 initialized with `render_mode=None` is marked as deprecated and will be removed in the future.
  logger.deprecation(
/usr/local/lib/python3.11/dist-packages/gym/wrappers/monitoring/video_recorder.py:101: DeprecationWarning: WARN: <class 'gym.wrappers.monitoring.video_recorder.VideoRecorder'> is marked as deprecated and will be removed in the future.
  logger.deprecation(
/usr/local/lib/python3.11/dist-packages/gym/wrappers/monitoring/video_recorder.py:149: DeprecationWarning: WARN: `env.metadata["video.frames_per_second"] is

# Grabar video de agente aleatorio
Crea un nuevo entorno SuperMarioBros-1-1-v0 configurado con preprocesamiento (escala de grises, redimensionamiento y apilamiento de frames), y graba un video en el que Mario actúa aleatoriamente durante un máximo de 1000 pasos o hasta que termine el episodio. El video resultante se guarda y se muestra directamente en el notebook mediante HTML embebido. Ideal para comparar el comportamiento sin entrenamiento (aleatorio) frente al agente entrenado.

In [15]:
import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from gym.wrappers import GrayScaleObservation, ResizeObservation
from gym.wrappers.frame_stack import FrameStack
from gym.wrappers import TimeLimit

import base64, io
from IPython.display import HTML, display
from gym.wrappers.monitoring.video_recorder import VideoRecorder

# ✅ Crear nuevo entorno con render_mode
env_random = gym_super_mario_bros.make("SuperMarioBros-1-1-v0")
env_random = JoypadSpace(env_random, SIMPLE_MOVEMENT)
env_random = GrayScaleObservation(env_random)
env_random = ResizeObservation(env_random, shape=84)
env_random = FrameStack(env_random, num_stack=4)
env_random = TimeLimit(env_random, max_episode_steps=5000)

# ✅ Iniciar grabación
video_path = "/kaggle/working/video_random.mp4"
recorder = VideoRecorder(env_random, video_path)

state = env_random.reset()
if isinstance(state, tuple): state = state[0]
recorder.capture_frame()

# ✅ Limitar duración del video (máx 1000 pasos)
max_steps = 1000
step_count = 0

while step_count < max_steps:
    action = env_random.action_space.sample()
    next_state, reward, done, info = env_random.step(action)
    if isinstance(next_state, tuple): next_state = next_state[0]
    recorder.capture_frame()
    state = next_state
    step_count += 1
    if done or info.get("flag_get", False):
        break

recorder.close()

# ✅ Mostrar el video grabado
def show_video_random(path):
    with io.open(path, 'r+b') as f:
        video = f.read()
    encoded = base64.b64encode(video).decode('ascii')
    return HTML(f"""
        <video width=480 controls autoplay loop>
            <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
        </video>
    """)

print("🎲 Partida aleatoria grabada.")
display(show_video_random(video_path))


/usr/local/lib/python3.11/dist-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment SuperMarioBros-1-1-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(
/usr/local/lib/python3.11/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.11/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.11/dist-packages/gym/wrappers/monitoring/video_recorder.py:67: DeprecationWarning: WARN: `env.metadata["render.modes"] is marked as deprecated and will be replaced with `

🎲 Partida aleatoria grabada.
